In [5]:
# 将adam_u中2dof头替换到adam_sp上,产出adam_sp_pro.urdf

import os
import shutil
import xml.etree.ElementTree as ET

REPLACE_LINKS_NAME = ["torso"]
ADD_LINKS_NAME = ["neckYaw_link", "neckPitch_link"]
ADD_JOINTS_NAME = ["neckYaw", "neckPitch"]
assert len(ADD_LINKS_NAME) == len(ADD_JOINTS_NAME)

u_tree = ET.parse("../adam_u/adam_u.urdf")
u_root = u_tree.getroot()

sp_tree = ET.parse("../adam_sp/adam_sp.urdf")
sp_root = sp_tree.getroot()

# for link in REPLACE_LINKS_NAME:
#     u_link = u_root.find(f".//link[@name='{link}']")
#     sp_link = sp_root.find(f".//link[@name='{link}']")
#     idx = list(sp_root).index(sp_link)
#     if u_link is not None and sp_link is not None:
#         sp_root.remove(sp_link)
#         sp_root.insert(idx, u_link)
#         print(f"Replacing link: {idx} {link}")
#     else:
#         print(f"ERROR: Link {link} not found in one of the files.")
#         exit(1)

for link, joint in zip(ADD_LINKS_NAME, ADD_JOINTS_NAME):
    u_link = u_root.find(f".//link[@name='{link}']")
    u_joint = u_root.find(f".//joint[@name='{joint}']")
    if u_link is not None and u_joint is not None:
        sp_root.append(u_link)
        print(f"Adding link: {link}")
        sp_root.append(u_joint)
        print(f"Adding joint: {joint}")
    else:
        print(f"ERROR: Link {link} or joint {joint} not found in adam_u.urdf.")
        exit(1)

new_folder = "../adam_sp_pro/"
if os.path.exists(new_folder):
    shutil.rmtree(new_folder)
os.makedirs(new_folder, exist_ok=True)
sp_root.attrib["name"] = "adam_sp_pro"
ET.indent(sp_tree, space="  ")
sp_tree.write("../adam_sp_pro/adam_sp_pro.urdf", encoding="utf-8", xml_declaration=True)


world = ET.Element("link", name="world")
sp_root.insert(0, world)
floating = ET.Element("joint", name="floating_joint", type="floating")
ET.SubElement(floating, "parent", link="world")
ET.SubElement(floating, "child", link="pelvis")
ET.SubElement(floating, "origin", xyz="0 0 0", rpy="0 0 0")
ET.SubElement(floating, "axis", xyz="0 0 0 1 0 0")
ET.SubElement(floating, "axis", xyz="0 0 0 0 1 0")
ET.SubElement(floating, "axis", xyz="0 0 0 0 0 1")
ET.SubElement(floating, "axis", xyz="1 0 0")
ET.SubElement(floating, "axis", xyz="0 1 0")
ET.SubElement(floating, "axis", xyz="0 0 1")
imu_idx = list(sp_root).index(sp_root.find(".//link[@name='imu_link']"))
sp_root.insert(imu_idx, floating)
ET.indent(sp_tree, space="  ")
sp_tree.write("../adam_sp_pro/adam_sp_pro_for_mpc.urdf", encoding="utf-8", xml_declaration=True)

shutil.copytree("meshes", "../adam_sp_pro/meshes")
shutil.copytree("../adam_u/meshes_dae", "../adam_sp_pro/meshes_dae")
shutil.copytree("../adam_u/meshes_stl_0.25", "../adam_sp_pro/meshes_stl_0.25")

Adding link: neckYaw_link
Adding joint: neckYaw
Adding link: neckPitch_link
Adding joint: neckPitch


'../adam_sp_pro/meshes_stl_0.25'

In [6]:
from urdf_parser_py.urdf import URDF

# 从文件加载 URDF
robot = URDF.from_xml_file("../adam_sp_pro/adam_sp_pro.urdf")

# 打印基本信息
print(f"机器人名称: {robot.name}")
print(f"链接数量: {len(robot.links)}")
print(f"关节数量: {len(robot.joints)}")


机器人名称: adam_sp_pro
链接数量: 57
关节数量: 56
